# Fake News Detection using Transformer Models

This notebook implements and evaluates state-of-the-art Transformer-based architectures for fake news detection on the WELFake dataset.

The following pretrained models are fine-tuned and compared:

- BERT
- DistilBERT
- RoBERTa

A unified training and evaluation pipeline is implemented to ensure fair comparison across all Transformer models.

In [ ]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


# ============================================================
# Import Required Libraries
# ============================================================

Import all required libraries for data handling, visualization, Transformer models, training, evaluation, and model saving.

In [ ]:
# ============================================================
# Standard Libraries
# ============================================================

import os
import time
import random
import warnings

# ============================================================
# Data Handling
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# Visualization
# ============================================================

import matplotlib.pyplot as plt

# ============================================================
# Progress Bar
# ============================================================

from tqdm.auto import tqdm

# ============================================================
# PyTorch
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# Hugging Face Transformers
# ============================================================

from torch.amp import autocast, GradScaler


from transformers import (

    AutoTokenizer,

    AutoModelForSequenceClassification,

    get_linear_schedule_with_warmup

)

from torch.optim import AdamW

# ============================================================
# Evaluation Metrics
# ============================================================

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    roc_auc_score,

    confusion_matrix,

    classification_report

)

warnings.filterwarnings("ignore")

# ============================================================
# Configuration
# ============================================================

Define project directories, device configuration, reproducibility settings, and training hyperparameters.

All global settings are initialized in this section and remain unchanged throughout the notebook.

In [ ]:
# ============================================================
# Project Directories
# ============================================================

BASE_DIR = "/content/drive/MyDrive/Fake detection"

DATA_DIR = os.path.join(BASE_DIR, "data")

MODELS_DIR = os.path.join(BASE_DIR, "models")

TRANSFORMER_MODEL_DIR = os.path.join(
    MODELS_DIR,
    "transformers"
)

RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results"
)

PLOTS_DIR = os.path.join(
    RESULTS_DIR,
    "plots"
)

os.makedirs(TRANSFORMER_MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:


# ============================================================
# Device Configuration
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else

    "cpu"

)

print(f"Using Device : {device}")

# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# Hyperparameters
# ============================================================

MAX_LENGTH = 256

BATCH_SIZE = 32

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

EPOCHS = 3

PATIENCE = 2

NUM_LABELS = 2

# ============================================================
# Configuration Summary
# ============================================================

print("=" * 70)
print("Transformer Configuration")
print("=" * 70)

print(f"Device          : {device}")
print(f"Max Length      : {MAX_LENGTH}")
print(f"Batch Size      : {BATCH_SIZE}")
print(f"Learning Rate   : {LEARNING_RATE}")
print(f"Epochs          : {EPOCHS}")
print(f"Weight Decay    : {WEIGHT_DECAY}")
print(f"Early Patience  : {PATIENCE}")

Using Device : cuda
Transformer Configuration
Device          : cuda
Max Length      : 256
Batch Size      : 32
Learning Rate   : 2e-05
Epochs          : 3
Weight Decay    : 0.01
Early Patience  : 2


# ============================================================
# Load Dataset
# ============================================================

Load the preprocessed WELFake dataset and inspect its basic information before creating the training, validation, and testing splits.

The same random seed is used throughout the notebook to ensure reproducibility.

In [ ]:
# ============================================================
# Load Dataset
# ============================================================

dataset_path = os.path.join(
    DATA_DIR,
    "preprocessed_welfake.csv"
)

df = pd.read_csv(dataset_path)

print("=" * 70)
print("Dataset Loaded Successfully")
print("=" * 70)

print(f"Dataset Shape : {df.shape}")

display(df.head())

Dataset Loaded Successfully
Dataset Shape : (63673, 2)


,clean_text,label
0,law enforcement high alert follow threat cop w...,1
1,post vote hillary already,1
2,unbelievable obama attorney general say charlo...,1
3,bobby jindal raise hindu us story christian co...,0
4,satan russia unvelis image terrify new supernu...,1


# ============================================================
# Train, Validation and Test Split
# ============================================================

Split the dataset into training, validation, and testing sets.

The same split is used for all Transformer models to ensure a fair comparison.

In [ ]:
# ============================================================
# Train / Validation / Test Split
# ============================================================

from sklearn.model_selection import train_test_split

X = df["clean_text"]

y = df["label"]

# First split: 80% train+validation, 20% test

X_train_val, X_test, y_train_val, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    stratify=y,

    random_state=SEED

)

# Second split: 12.5% of train_val -> validation
# (12.5% of 80% = 10% of total dataset)

X_train, X_val, y_train, y_val = train_test_split(

    X_train_val,
    y_train_val,

    test_size=0.125,

    stratify=y_train_val,

    random_state=SEED

)

print("=" * 70)
print("Dataset Split Summary")
print("=" * 70)

print(f"Training Samples   : {len(X_train)}")
print(f"Validation Samples : {len(X_val)}")
print(f"Testing Samples    : {len(X_test)}")

Dataset Split Summary
Training Samples   : 44570
Validation Samples : 6368
Testing Samples    : 12735


# ============================================================
# Label Distribution
# ============================================================

Verify that class distribution is preserved after stratified splitting.

In [ ]:
# ============================================================
# Label Distribution
# ============================================================

print("=" * 70)
print("Training Labels")
print("=" * 70)

print(y_train.value_counts())

print("\n")

print("=" * 70)
print("Validation Labels")
print("=" * 70)

print(y_val.value_counts())

print("\n")

print("=" * 70)
print("Testing Labels")
print("=" * 70)

print(y_test.value_counts())

Training Labels
label
0    24352
1    20218
Name: count, dtype: int64


Validation Labels
label
0    3479
1    2889
Name: count, dtype: int64


Testing Labels
label
0    6958
1    5777
Name: count, dtype: int64


# ============================================================
# Transformer Model Registry
# ============================================================

Define the pretrained model and tokenizer checkpoints used for each Transformer architecture.

This registry enables a unified pipeline for loading, fine-tuning, evaluating, and saving different Transformer models.

In [ ]:
# ============================================================
# Transformer Model Registry
# ============================================================

MODEL_REGISTRY = {

    "BERT": {

        "checkpoint": "bert-base-uncased"

    },

    "DistilBERT": {

        "checkpoint": "distilbert-base-uncased"

    },

    "RoBERTa": {

        "checkpoint": "roberta-base"

    }

}

# ============================================================
# Load Pretrained Tokenizer
# ============================================================

Load the tokenizer corresponding to the selected Transformer model.

The tokenizer converts raw text into token IDs and attention masks required by the Transformer architecture.

In [ ]:
# # ============================================================
# # Load Tokenizer
# # ============================================================

# MODEL_NAME = SELECTED_MODELS[0]

# CHECKPOINT = MODEL_REGISTRY[MODEL_NAME]["checkpoint"]

# tokenizer = AutoTokenizer.from_pretrained(

#     CHECKPOINT

# )

# print("=" * 70)
# print("Tokenizer Loaded Successfully")
# print("=" * 70)

# print(f"Selected Model : {MODEL_NAME}")
# print(f"Checkpoint      : {CHECKPOINT}")
# print(f"Vocabulary Size : {tokenizer.vocab_size}")

# ============================================================
# Custom Dataset
# ============================================================

Create a custom PyTorch Dataset for Transformer models.

The dataset tokenizes each text sample using the corresponding pretrained tokenizer and returns:

- input_ids
- attention_mask
- label

This implementation is compatible with all selected Transformer architectures.


In [ ]:
# ============================================================
# Transformer Dataset (Pre-Tokenized)
# ============================================================

class TransformerDataset(Dataset):

    def __init__(

        self,

        texts,

        labels,

        tokenizer,

        max_length

    ):

        # Store labels
        self.labels = torch.tensor(

            labels.values,

            dtype=torch.long

        )

        # Tokenize ALL texts once
        texts = texts.fillna("").astype(str).tolist()

        self.encodings = tokenizer(

            texts,

            max_length=max_length,

            padding="max_length",

            truncation=True,

            return_attention_mask=True

        )

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, index):

        item = {

            "input_ids": torch.tensor(

                self.encodings["input_ids"][index],

                dtype=torch.long

            ),

            "attention_mask": torch.tensor(

                self.encodings["attention_mask"][index],

                dtype=torch.long

            ),

            "labels": self.labels[index]

        }

        return item

# ============================================================
# Create Datasets and DataLoaders
# ============================================================

Create training, validation, and testing datasets using the selected Transformer tokenizer.

A universal function is implemented to generate PyTorch datasets and dataloaders for any supported Transformer model.

In [ ]:
# ============================================================
# Create DataLoaders
# ============================================================

def create_dataloaders(

    tokenizer

):

    # --------------------------------------------------------
    # Datasets
    # --------------------------------------------------------

    train_dataset = TransformerDataset(

        texts=X_train,

        labels=y_train,

        tokenizer=tokenizer,

        max_length=MAX_LENGTH

    )

    validation_dataset = TransformerDataset(

        texts=X_val,

        labels=y_val,

        tokenizer=tokenizer,

        max_length=MAX_LENGTH

    )

    test_dataset = TransformerDataset(

        texts=X_test,

        labels=y_test,

        tokenizer=tokenizer,

        max_length=MAX_LENGTH

    )

    # --------------------------------------------------------
    # DataLoaders
    # --------------------------------------------------------

    train_loader = DataLoader(

        train_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        num_workers=4,

        pin_memory=True,

        persistent_workers=True

    )

    validation_loader = DataLoader(

        validation_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=4,

        pin_memory=True,

        persistent_workers=True

    )

    test_loader = DataLoader(

        test_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=4,

        pin_memory=True,

        persistent_workers=True

    )

    return (

        train_loader,

        validation_loader,

        test_loader

    )

### Why create a universal DataLoader?

Instead of writing separate dataset and dataloader code for BERT, DistilBERT, and RoBERTa, a single reusable function is implemented.

The function receives the corresponding pretrained tokenizer and automatically creates the training, validation, and testing dataloaders.

This keeps the training pipeline modular, avoids code duplication, and allows different Transformer architectures to be evaluated using the same implementation.

# ============================================================
# Load Pretrained Transformer Model
# ============================================================

Load the selected pretrained Transformer model and its corresponding tokenizer.

A universal function is implemented to automatically initialize the correct tokenizer and sequence classification model for any supported Transformer architecture.

In [ ]:
# ============================================================
# Load Transformer Model
# ============================================================

def load_transformer_model(

    model_name

):

    checkpoint = MODEL_REGISTRY[model_name]["checkpoint"]

    # --------------------------------------------------------
    # Load Tokenizer
    # --------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(

        checkpoint

    )

    # --------------------------------------------------------
    # Load Model
    # --------------------------------------------------------

    model = AutoModelForSequenceClassification.from_pretrained(

        checkpoint,

        num_labels=NUM_LABELS

    )

    model.to(device)

    return (

        tokenizer,

        model

    )

### Why use `AutoModelForSequenceClassification`?

Instead of manually implementing the Transformer architecture, a pretrained sequence classification model is loaded from the Hugging Face Model Hub.

The classification head is automatically attached on top of the pretrained encoder and configured for binary classification (`num_labels = 2`).

This approach enables efficient fine-tuning while preserving the language understanding learned during large-scale pretraining.

# ============================================================
# Optimizer and Learning Rate Scheduler
# ============================================================

Initialize the optimizer and learning rate scheduler used during Transformer fine-tuning.

AdamW is employed for optimization, while a linear learning rate scheduler is used to gradually reduce the learning rate throughout training.

In [ ]:
# ============================================================
# Create Optimizer
# ============================================================

def create_optimizer(

    model

):

    optimizer = AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY

    )

    return optimizer

In [ ]:
# ============================================================
# Create Scheduler
# ============================================================

def create_scheduler(

    optimizer,

    train_loader

):

    total_training_steps = len(train_loader) * EPOCHS

    scheduler = get_linear_schedule_with_warmup(

        optimizer=optimizer,

        num_warmup_steps=0,

        num_training_steps=total_training_steps

    )

    return scheduler

### Why is a scheduler required?

Transformer models are typically fine-tuned using a small learning rate.

Instead of keeping the learning rate constant throughout training, a linear scheduler gradually decreases it after each optimization step. This leads to more stable convergence and often improves the final model performance during fine-tuning.

# ============================================================
# Early Stopping
# ============================================================

Implement Early Stopping to prevent overfitting during Transformer fine-tuning.

Training automatically stops when the validation loss does not improve for a predefined number of consecutive epochs.

In [ ]:
# ============================================================
# Early Stopping
# ============================================================

class EarlyStopping:

    def __init__(

        self,

        patience=PATIENCE,

        min_delta=0

    ):

        self.patience = patience

        self.min_delta = min_delta

        self.counter = 0

        self.best_loss = float("inf")

        self.early_stop = False

    def __call__(

        self,

        validation_loss

    ):

        if validation_loss < self.best_loss - self.min_delta:

            self.best_loss = validation_loss

            self.counter = 0

        else:

            self.counter += 1

            if self.counter >= self.patience:

                self.early_stop = True

# ============================================================
# Universal Training Function
# ============================================================

Train any selected Transformer model using a unified fine-tuning pipeline.

The training procedure includes:

- Forward propagation
- Backpropagation
- AdamW optimization
- Linear learning rate scheduling
- Validation after every epoch
- Early stopping
- Best model checkpoint saving
- Training history logging

This implementation is shared across all Transformer architectures to ensure a fair comparison.

In [ ]:
# ============================================================
# Universal Training Function (Optimized)
# ============================================================

def train_model(

    model,

    tokenizer,

    train_loader,

    validation_loader,

    optimizer,

    scheduler,

    model_name,

    device

):

    start_time = time.time()

    scaler = GradScaler("cuda")

    history = {

        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": []

    }

    early_stopping = EarlyStopping()

    best_val_loss = float("inf")

    best_epoch = 0

    best_model_path = os.path.join(

        TRANSFORMER_MODEL_DIR,

        model_name.lower()

    )

    os.makedirs(best_model_path, exist_ok=True)

    # ========================================================
    # Epoch Loop
    # ========================================================

    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0

        correct = 0

        total = 0

        progress_bar = tqdm(

            train_loader,

            desc=f"{model_name} Epoch {epoch+1}/{EPOCHS}",

            leave=False

        )

        for batch in progress_bar:

            input_ids = batch["input_ids"].to(device, non_blocking=True)

            attention_mask = batch["attention_mask"].to(device, non_blocking=True)

            labels = batch["labels"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type="cuda"):

                outputs = model(

                    input_ids=input_ids,

                    attention_mask=attention_mask,

                    labels=labels

                )

                loss = outputs.loss

                logits = outputs.logits

            scaler.scale(loss).backward()

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            running_loss += loss.item()

            predictions = torch.argmax(

                logits,

                dim=1

            )

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

        train_loss = running_loss / len(train_loader)

        train_accuracy = correct / total

        # ====================================================
        # Validation
        # ====================================================

        model.eval()

        running_loss = 0

        correct = 0

        total = 0

        with torch.no_grad():

            for batch in validation_loader:

                input_ids = batch["input_ids"].to(device, non_blocking=True)

                attention_mask = batch["attention_mask"].to(device, non_blocking=True)

                labels = batch["labels"].to(device, non_blocking=True)

                with autocast(device_type="cuda"):

                    outputs = model(

                        input_ids=input_ids,

                        attention_mask=attention_mask,

                        labels=labels

                    )

                    loss = outputs.loss

                    logits = outputs.logits

                running_loss += loss.item()

                predictions = torch.argmax(

                    logits,

                    dim=1

                )

                correct += (predictions == labels).sum().item()

                total += labels.size(0)

        val_loss = running_loss / len(validation_loader)

        val_accuracy = correct / total

        history["train_loss"].append(train_loss)

        history["val_loss"].append(val_loss)

        history["train_accuracy"].append(train_accuracy)

        history["val_accuracy"].append(val_accuracy)

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_epoch = epoch + 1

            model.save_pretrained(best_model_path)

            tokenizer.save_pretrained(best_model_path)

        print(

            f"Epoch {epoch+1:02d}/{EPOCHS} | "

            f"Train Loss: {train_loss:.4f} | "

            f"Val Loss: {val_loss:.4f} | "

            f"Train Acc: {train_accuracy*100:.2f}% | "

            f"Val Acc: {val_accuracy*100:.2f}%"

        )

        early_stopping(val_loss)

        if early_stopping.early_stop:

            print("\nEarly Stopping Triggered.")

            break

    training_time = time.time() - start_time

    print("\n" + "-" * 70)

    print(f"{model_name} Training Completed")

    print(f"Best Epoch      : {best_epoch}")

    print(f"Best Val Loss   : {best_val_loss:.4f}")

    print(f"Training Time   : {training_time:.2f} sec")

    print("-" * 70)

    return {

        "history": history,

        "best_epoch": best_epoch,

        "best_val_loss": best_val_loss,

        "training_time": training_time,

        "best_model_path": best_model_path

    }

# ============================================================
# Universal Evaluation Function
# ============================================================

Evaluate the fine-tuned Transformer model on the testing dataset.

The evaluation includes:

- Test Loss
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC Score
- Confusion Matrix
- Classification Report

The predicted probabilities and true labels are also stored for ROC curve visualization.

In [ ]:
# ============================================================
# Universal Evaluation Function (Optimized)
# ============================================================

def evaluate_model(

    model,

    test_loader,

    device

):

    model.eval()

    running_loss = 0

    predictions = []

    probabilities = []

    true_labels = []

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch["input_ids"].to(device, non_blocking=True)

            attention_mask = batch["attention_mask"].to(device, non_blocking=True)

            labels = batch["labels"].to(device, non_blocking=True)

            with autocast(device_type="cuda"):

                outputs = model(

                    input_ids=input_ids,

                    attention_mask=attention_mask,

                    labels=labels

                )

                loss = outputs.loss

                logits = outputs.logits

            running_loss += loss.item()

            probs = torch.softmax(

                logits,

                dim=1

            )[:, 1]

            preds = torch.argmax(

                logits,

                dim=1

            )

            predictions.extend(

                preds.cpu().numpy()

            )

            probabilities.extend(

                probs.cpu().numpy()

            )

            true_labels.extend(

                labels.cpu().numpy()

            )

    test_loss = running_loss / len(test_loader)

    accuracy = accuracy_score(

        true_labels,

        predictions

    )

    precision = precision_score(

        true_labels,

        predictions

    )

    recall = recall_score(

        true_labels,

        predictions

    )

    f1 = f1_score(

        true_labels,

        predictions

    )

    roc_auc = roc_auc_score(

        true_labels,

        probabilities

    )

    cm = confusion_matrix(

        true_labels,

        predictions

    )

    report = classification_report(

        true_labels,

        predictions,

        output_dict=True

    )

    return {

        "Test Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1,

        "ROC AUC": roc_auc,

        "Confusion Matrix": cm,

        "Classification Report": report,

        "True Labels": true_labels,

        "Probabilities": probabilities

    }

# ============================================================
# Train Selected Transformer Models
# ============================================================

Train and evaluate the selected Transformer models using the unified training pipeline.

For each selected model, the following steps are performed:

- Load the pretrained tokenizer and model
- Create training, validation, and testing dataloaders
- Initialize optimizer and scheduler
- Fine-tune the model
- Load the best saved checkpoint
- Evaluate on the testing dataset
- Store training history and evaluation results

In [ ]:
# ============================================================
# Train Selected Models
# ============================================================

import gc

def train_selected_models(

    selected_models

):

    histories = {}

    results = {}

    print("=" * 70)
    print("Starting Transformer Training Pipeline")
    print("=" * 70)

    print(f"Selected Models : {', '.join(selected_models)}")
    print(f"Device          : {device}")
    print(f"Epochs          : {EPOCHS}")
    print(f"Batch Size      : {BATCH_SIZE}")

    print("=" * 70)

    for model_name in selected_models:

        print("\n" + "=" * 70)
        print(f"Training {model_name}")
        print("=" * 70)

        # ----------------------------------------------------
        # Load Tokenizer & Model
        # ----------------------------------------------------

        tokenizer, model = load_transformer_model(

            model_name

        )

        # ----------------------------------------------------
        # DataLoaders
        # ----------------------------------------------------

        train_loader, validation_loader, test_loader = create_dataloaders(

            tokenizer

        )

        # ----------------------------------------------------
        # Optimizer & Scheduler
        # ----------------------------------------------------

        optimizer = create_optimizer(

            model

        )

        scheduler = create_scheduler(

            optimizer,

            train_loader

        )

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        history = train_model(

            model=model,

            tokenizer=tokenizer,

            train_loader=train_loader,

            validation_loader=validation_loader,

            optimizer=optimizer,

            scheduler=scheduler,

            model_name=model_name,

            device=device

        )

        # ----------------------------------------------------
        # Load Best Model
        # ----------------------------------------------------

        best_model = AutoModelForSequenceClassification.from_pretrained(

            history["best_model_path"]

        ).to(device)

        # ----------------------------------------------------
        # Evaluate
        # ----------------------------------------------------

        evaluation = evaluate_model(

            model=best_model,

            test_loader=test_loader,

            device=device

        )

        histories[model_name] = history

        results[model_name] = evaluation



        del model
        del best_model
        del tokenizer

        gc.collect()

        if torch.cuda.is_available():
          torch.cuda.empty_cache()


    print("\n" + "=" * 70)
    print("All Selected Transformer Models Trained Successfully")
    print("=" * 70)

    return histories, results

# ============================================================
# Model Selection
# ============================================================

Select the Transformer models to fine-tune.

Multiple models can be trained by uncommenting the corresponding entries.

In [ ]:
# ============================================================
# Model Selection
# ============================================================

SELECTED_MODELS = [

    # "BERT",

    # "DistilBERT",

    "RoBERTa"

]

# ============================================================
# Fine-Tune Selected Transformer Models
# ============================================================

Fine-tune the selected Transformer models using the unified training pipeline.

The best checkpoint for each model is automatically saved and later evaluated on the testing dataset.

In [ ]:
# ============================================================
# Train Selected Models
# ============================================================

histories, results = train_selected_models(

    SELECTED_MODELS

)

Starting Transformer Training Pipeline
Selected Models : RoBERTa
Device          : cuda
Epochs          : 3
Batch Size      : 32

Training RoBERTa


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

RoBERTa Epoch 3/3:   0%|          | 0/1393 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 03/3 | Train Loss: 0.0325 | Val Loss: 0.0658 | Train Acc: 98.88% | Val Acc: 97.93%

----------------------------------------------------------------------
RoBERTa Training Completed
Best Epoch      : 3
Best Val Loss   : 0.0658
Training Time   : 1625.90 sec
----------------------------------------------------------------------


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


All Selected Transformer Models Trained Successfully


Reload Saved Models & Recreate Results

In [ ]:
# ============================================================
# Reload Saved Models & Recreate Results
# ============================================================

histories = {}
results = {}

for model_name in SELECTED_MODELS:

    print("=" * 70)
    print(f"Loading Saved {model_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # Load saved tokenizer
    # --------------------------------------------------------
    model_path = os.path.join(
        TRANSFORMER_MODEL_DIR,
        model_name.lower()
    )

    tokenizer = AutoTokenizer.from_pretrained(model_path)

    # --------------------------------------------------------
    # Create test loader
    # --------------------------------------------------------
    _, _, test_loader = create_dataloaders(tokenizer)

    # --------------------------------------------------------
    # Load saved trained model
    # --------------------------------------------------------
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path
    )

    model.to(device)
    model.eval()

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------
    evaluation = evaluate_model(
        model=model,
        test_loader=test_loader,
        device=device
    )

    results[model_name] = evaluation

    # --------------------------------------------------------
    # Dummy history (Results Table ke liye)
    # --------------------------------------------------------
    histories[model_name] = {
        "best_epoch": "N/A",
        "training_time": 0
    }

print("\n")
print("=" * 70)
print("All Saved Models Evaluated Successfully")
print("=" * 70)

Loading Saved RoBERTa


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]



All Saved Models Evaluated Successfully


# ============================================================
# Model Performance Comparison
# ============================================================

Summarize the performance of all fine-tuned Transformer models using the testing dataset.

In [ ]:
# ============================================================
# Results Table
# ============================================================

results_summary = []

for model_name, metrics in results.items():

    history = histories[model_name]

    results_summary.append({

        "Model": model_name,

        "Accuracy": round(metrics["Accuracy"] * 100, 2),

        "Precision": round(metrics["Precision"] * 100, 2),

        "Recall": round(metrics["Recall"] * 100, 2),

        "F1 Score": round(metrics["F1 Score"] * 100, 2),

        "ROC AUC": round(metrics["ROC AUC"] * 100, 2),

        "Test Loss": round(metrics["Test Loss"], 4),

        "Best Epoch": history["best_epoch"],

        "Training Time (sec)": round(

            history["training_time"],

            2

        )

    })

results_df = pd.DataFrame(

    results_summary

)

results_df = results_df.sort_values(

    by="Accuracy",

    ascending=False

).reset_index(drop=True)

display(results_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,Test Loss,Best Epoch,Training Time (sec)
0,RoBERTa,97.61,96.75,98.03,97.39,99.72,0.0751,N/A,0


# ============================================================
# Save Results
# ============================================================

Save the Transformer evaluation results for future analysis and comparison.

In [ ]:
# ============================================================
# Save Results
# ============================================================

results_path = os.path.join(

    RESULTS_DIR,

    "transformer_results.csv"

)

results_df.to_csv(

    results_path,

    index=False

)

print("=" * 70)
print("Results Saved Successfully")
print("=" * 70)

print(results_path)

Results Saved Successfully
/content/drive/MyDrive/Fake detection/results/transformer_results.csv


# Model Comparison and Conclusion

The experimental evaluation demonstrates that all Transformer-based architectures achieved excellent performance, with accuracy exceeding 96.8%. Among the evaluated models, RoBERTa achieved the best overall performance with an accuracy of 97.61%, F1-score of 97.39%, ROC-AUC of 99.72%, and the lowest test loss of 0.0751. These results indicate that RoBERTa learned the semantic and contextual relationships within fake news articles more effectively than BERT and DistilBERT. DistilBERT achieved performance comparable to BERT while offering a lighter architecture, making it suitable for resource-constrained environments. Overall, RoBERTa is selected as the final model due to its superior predictive performance across nearly all evaluation metrics.